# Гир 1 — фиксированная стратегия + симуляция

Офлайн-бэктест спреда Bybit/OKX по parquet: пороги входа/выхода, опциональные гейты свежести/латентности/среднего, метрика closed trades `sum(open_spread + close_spread)` (с учётом scale/fees, если включены). **Не Optuna** — только один прогон с заданными параметрами.

## Приближения

- **Равная задержка исполнения на обеих биржах:** `Trade_Lat` (мс) одна на Bybit и OKX. При сигнале fill берётся не с тика сигнала, а с первого тика, где `event_local_ts_ms >= signal_ts + Trade_Lat` (причинно). `Trade_Lat = 0` → fill на тике сигнала (сравнение с baseline).
- **Гейт объёма выключен заглушкой:** `Check_volume = False`. Логика размера стакана (как в `else/bybit_ws.py`) написана, но не применяется. Когда появятся полные колонки size — включить `Check_volume = True`.
- Позиция по умолчанию полная (`position_frac=1` → quantity=100); комиссии `fee_rate=0`, пока не включены явно.
- Метрика open+close по знаку формулы не меняется; scale/fees согласованы с delayed fill.

## Параметры

**Variation (позже оптимизация):** `thresh_open_*`, `thresh_close_*`, `open_frac`, `close_frac`.

**Fixed:** `max_latency_*_ms`, `avg_window_sec`, `max_freshness_ms`, `Trade_Lat`, `Check_volume`, `position_size`, `position_frac`, `fee_rate`.

## Гейты

- Gate A: freshness (optional) + per-leg latency caps на тике сигнала
- Gate B: среднее спреда в окне `avg_window_sec` только по точкам с OKX≤cap_okx **и** Bybit≤cap_bybit
- Gate volume (`Check_volume`): размер стакана на торгуемых сторонах open/close по аналогии с ботом (Bybit > `position_size`, OKX > `position_size/10`)


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Literal, Optional

import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- data / plot config ---
base_coin = "LA"
event_date = "2026-07-22"
# Max points for line traces (trade markers stay on full timestamps).
max_points = 4000
# Histogram bins for latency distribution plot.
latency_hist_bins = 60

# =============================================================================
# VARIATION PARAMS — demo defaults; future optimizer search space
# (thresholds + open_frac / close_frac)
# =============================================================================
thresh_open_long = 0.5
thresh_open_short = 0.5
thresh_close_long = 0.5
thresh_close_short = 0.5
open_frac = 0.8  # mean_spread >= open_frac * thresh_open_*  (Gate B)
close_frac = 0.8  # mean_spread >= close_frac * thresh_close_* (Gate B, symmetric)

# Demo comparison fracs for gated ON run (cell 3); OFF uses ungated defaults above.
demo_open_frac = 0.7
demo_close_frac = 0.7

# =============================================================================
# FIXED HYPERPARAMS — set per run; NOT in variation space at this stage
# =============================================================================
# Gate A freshness (optional fixed; user did not put it in variation).
max_freshness_ms = None  # e.g. 500.0; both okx/bybit_freshness_ms <= cap
# Gate A / Gate B per-leg latency caps (None = that leg unchecked / no filter).
max_latency_okx_ms = 200  # e.g. 150.0; okx_latency_ms <= cap
max_latency_bybit_ms = 200  # e.g. 150.0; bybit_latency_ms <= cap
# Gate B window width (None disables Gate B).
avg_window_sec = None  # e.g. 2.0

# Demo fixed hyperparams for gated ON comparison (cell 3).
demo_max_freshness_ms = 500.0
demo_max_latency_okx_ms = 150.0
demo_max_latency_bybit_ms = 150.0
demo_avg_window_sec = 2.0

# =============================================================================
# GEAR-1 EXECUTION / VOLUME — fixed hyperparams
# =============================================================================
# Equal execution delay on BOTH exchanges (ms). Fill from first tick with
# event_local_ts_ms >= signal_ts + Trade_Lat. 0 = fill on signal tick (baseline).
Trade_Lat = 100

# Volume gate stub: False → always pass (required while size data incomplete).
# True → bybit_ws.py analogy on open/close sizes; missing size cols → hard fail.
Check_volume = False
position_size = 10.0  # Bybit coin units; OKX contract threshold = position_size / 10

# Optional realism knobs (defaults preserve baseline metric formula).
position_frac = 1.0          # (0,1]; 1.0 = full size (baseline quantity=100)
fee_rate = 0.0               # fraction of notional per fill leg (e.g. 0.0005 = 5 bps)

# Demo realism ON knobs (cell 3 comparison); baseline path keeps defaults above.
demo_position_frac = 0.5
demo_fee_rate = 0.0005       # 5 bps / leg → ~20 bps round-trip (4 legs)
demo_Check_volume = False    # keep False until size columns are complete
demo_Trade_Lat = 100

PARQUET_ROOT = Path(
    "/Users/mishatrubik/Desktop/spread/output/spreads_parquet_by_coins"
)
DATA_PATH = PARQUET_ROOT / f"base_coin={base_coin}" / f"event_date={event_date}"

df = pd.read_parquet(DATA_PATH)
print(f"loaded {base_coin} {event_date}: {len(df)} rows from {DATA_PATH}")
print(f"Trade_Lat={Trade_Lat} ms  Check_volume={Check_volume}  position_size={position_size}")
print(df.head())
print(df.columns.tolist())


In [ ]:
Side = Literal["long", "short"]
TradeStatus = Literal["closed", "open"]

# Ticks left/right of the signal row included in latency windows (±radius + signal).
LATENCY_WINDOW_RADIUS = 5

# Book size columns (parquet / screaner schema) used by Gate C.
# long open:  buy OKX ask + sell Bybit bid
# short open: sell OKX bid + buy Bybit ask
# close reverses the sides.
BOOK_SIZE_COLS = (
    "okx_bid_size",
    "okx_ask_size",
    "bybit_bid_size",
    "bybit_ask_size",
)


@dataclass
class Trade:
    """Single trade record. Open-at-end trades have status='open' and are excluded from metric.

    Scalar latency fields are parquet snapshots (`okx_latency_ms` / `bybit_latency_ms`)
    from the *signal* tick that initiated open or close — not a causal model.

    Window fields are positional neighborhoods of that signal tick in the sorted series:
    up to `LATENCY_WINDOW_RADIUS` ticks before (history for future SMA objectivity checks),
    the signal tick itself, and up to `LATENCY_WINDOW_RADIUS` ticks after (lookahead for
    post-signal latency / execution-delay analysis — research only, not live entry).
    Offsets are relative tick indices in [-radius, +radius]; lists shorten and
    `*_truncated=True` when the series edge clips the full window.

    Realism fields (optional; baseline leaves them 0/None/equal-to-signal):
    - `fees`: total fee cost in spread-% units deducted from pnl
    - `signal_*` vs fill prices/timestamps when exec delay shifts the fill
    """

    side: Side
    status: TradeStatus
    open_price: float
    open_ts: int
    open_dt: object
    quantity: float
    close_price: Optional[float] = None
    close_ts: Optional[int] = None
    close_dt: Optional[object] = None
    pnl: Optional[float] = None
    fees: float = 0.0
    # Delivery latency (ms) on the open-signal row
    okx_latency_ms_open: Optional[float] = None
    bybit_latency_ms_open: Optional[float] = None
    # Delivery latency (ms) on the close-signal row (None while still open)
    okx_latency_ms_close: Optional[float] = None
    bybit_latency_ms_close: Optional[float] = None
    # ±LATENCY_WINDOW_RADIUS neighborhood around open signal (len ≤ 2*radius+1)
    okx_latency_ms_open_window: Optional[list[Optional[float]]] = None
    bybit_latency_ms_open_window: Optional[list[Optional[float]]] = None
    latency_window_offsets_open: Optional[list[int]] = None
    latency_window_truncated_open: bool = False
    # Same neighborhood around close signal
    okx_latency_ms_close_window: Optional[list[Optional[float]]] = None
    bybit_latency_ms_close_window: Optional[list[Optional[float]]] = None
    latency_window_offsets_close: Optional[list[int]] = None
    latency_window_truncated_close: bool = False
    # Realism: signal tick vs fill tick (equal when delay disabled)
    signal_open_price: Optional[float] = None
    signal_open_ts: Optional[int] = None
    signal_close_price: Optional[float] = None
    signal_close_ts: Optional[int] = None
    open_fill_delay_ticks: int = 0
    close_fill_delay_ticks: int = 0


@dataclass
class BacktestResult:
    trades: list[Trade]
    metric: float
    open_position: Optional[Trade] = None
    # Observability: threshold candidates vs gates (open + close).
    n_signals_raw: int = 0
    n_signals_passed: int = 0
    n_filtered_by_freshness: int = 0
    n_filtered_by_latency: int = 0
    n_filtered_by_avg: int = 0
    n_filtered_by_size: int = 0
    n_pending_missed: int = 0  # scheduled fill past end of series
    fees_total: float = 0.0


def _latency_window_at(
    data: pd.DataFrame,
    i: int,
    *,
    radius: int = LATENCY_WINDOW_RADIUS,
) -> tuple[list[Optional[float]], list[Optional[float]], list[int], bool]:
    """Slice okx/bybit latency around positional index i: [i-radius, i+radius] clamped."""
    n = len(data)
    lo = max(0, i - radius)
    hi = min(n, i + radius + 1)
    truncated = lo > (i - radius) or hi < (i + radius + 1)
    offsets = list(range(lo - i, hi - i))

    def _vals(col: str) -> list[Optional[float]]:
        out: list[Optional[float]] = []
        for x in data[col].iloc[lo:hi].tolist():
            out.append(None if pd.isna(x) else float(x))
        return out

    return _vals("okx_latency_ms"), _vals("bybit_latency_ms"), offsets, truncated


def _fee_cost_pct(quantity: float, fee_rate: float, *, legs: int = 2) -> float:
    """Fee in spread-% units for `legs` fills, scaled by quantity/100.

    fee_rate is fraction of notional per leg (e.g. 0.0005). One leg → fee_rate*100
    percentage points; two legs on open (or close) → 2x. quantity=100 → full size.
    """
    if fee_rate == 0.0 or quantity == 0.0:
        return 0.0
    return float(fee_rate) * 100.0 * float(legs) * (float(quantity) / 100.0)


def _book_cols_for(side: Side, *, is_open: bool) -> tuple[str, str]:
    """Return (okx_size_col, bybit_size_col) for the traded book sides.

    Matches screaner legs and else/bybit_ws.py trade_manager size checks:
      long open:   okx_ask + bybit_bid
      short open:  okx_bid + bybit_ask
      close reverses the sides.
    """
    if is_open:
        if side == "long":
            return "okx_ask_size", "bybit_bid_size"
        return "okx_bid_size", "bybit_ask_size"
    # close reverses
    if side == "long":
        return "okx_bid_size", "bybit_ask_size"
    return "okx_ask_size", "bybit_bid_size"


def _require_size_columns(data: pd.DataFrame) -> None:
    """Hard-fail when Check_volume=True but size columns are missing."""
    missing = [c for c in BOOK_SIZE_COLS if c not in data.columns]
    if missing:
        raise ValueError(
            "Check_volume=True requires book size columns "
            f"{list(BOOK_SIZE_COLS)}, missing: {missing}. "
            "Set Check_volume=False until size data is available."
        )


def _volume_ok_bybit_ws(
    size_arrs: dict[str, np.ndarray],
    i: int,
    side: Side,
    *,
    is_open: bool,
    position_size: float,
) -> bool:
    """Size gate analogous to else/bybit_ws.py trade_manager.

    Bybit size must exceed `position_size` (coin units).
    OKX size must exceed `position_size / 10` (contract units in the bot).
    NaN sizes fail closed.
    """
    c_okx, c_bybit = _book_cols_for(side, is_open=is_open)
    s_okx = size_arrs[c_okx][i]
    s_bybit = size_arrs[c_bybit][i]
    if s_okx != s_okx or s_bybit != s_bybit:
        return False
    return float(s_bybit) > float(position_size) and float(s_okx) > (
        float(position_size) / 10.0
    )


def _resolve_fill_index(
    signal_i: int,
    ts_ms: np.ndarray,
    *,
    Trade_Lat: float,
) -> Optional[int]:
    """Causal fill index >= signal_i. None if delay cannot be satisfied in-series.

    Equal delay on both exchanges: first tick with
    event_local_ts_ms >= signal_ts + Trade_Lat. Trade_Lat <= 0 → signal_i.
    """
    n = len(ts_ms)
    delay = float(Trade_Lat)
    if delay <= 0:
        return signal_i
    target = float(ts_ms[signal_i]) + delay
    j = signal_i + 1
    while j < n and ts_ms[j] < target:
        j += 1
    return j if j < n else None


def _closed_trade(
    *,
    side: Side,
    open_price: float,
    open_ts: int,
    open_dt: object,
    close_price: float,
    close_ts: int,
    close_dt: object,
    quantity: float,
    fees: float,
    okx_latency_ms_open: Optional[float],
    bybit_latency_ms_open: Optional[float],
    okx_latency_ms_close: Optional[float],
    bybit_latency_ms_close: Optional[float],
    okx_latency_ms_open_window: Optional[list[Optional[float]]],
    bybit_latency_ms_open_window: Optional[list[Optional[float]]],
    latency_window_offsets_open: Optional[list[int]],
    latency_window_truncated_open: bool,
    okx_latency_ms_close_window: Optional[list[Optional[float]]],
    bybit_latency_ms_close_window: Optional[list[Optional[float]]],
    latency_window_offsets_close: Optional[list[int]],
    latency_window_truncated_close: bool,
    signal_open_price: Optional[float] = None,
    signal_open_ts: Optional[int] = None,
    signal_close_price: Optional[float] = None,
    signal_close_ts: Optional[int] = None,
    open_fill_delay_ticks: int = 0,
    close_fill_delay_ticks: int = 0,
) -> Trade:
    # Default score: scaled open+close spreads minus fees (baseline: scale=1, fees=0).
    pnl = (open_price + close_price) * (quantity / 100.0) - fees
    return Trade(
        side=side,
        status="closed",
        open_price=open_price,
        open_ts=open_ts,
        open_dt=open_dt,
        quantity=quantity,
        close_price=close_price,
        close_ts=close_ts,
        close_dt=close_dt,
        pnl=pnl,
        fees=fees,
        okx_latency_ms_open=okx_latency_ms_open,
        bybit_latency_ms_open=bybit_latency_ms_open,
        okx_latency_ms_close=okx_latency_ms_close,
        bybit_latency_ms_close=bybit_latency_ms_close,
        okx_latency_ms_open_window=okx_latency_ms_open_window,
        bybit_latency_ms_open_window=bybit_latency_ms_open_window,
        latency_window_offsets_open=latency_window_offsets_open,
        latency_window_truncated_open=latency_window_truncated_open,
        okx_latency_ms_close_window=okx_latency_ms_close_window,
        bybit_latency_ms_close_window=bybit_latency_ms_close_window,
        latency_window_offsets_close=latency_window_offsets_close,
        latency_window_truncated_close=latency_window_truncated_close,
        signal_open_price=signal_open_price if signal_open_price is not None else open_price,
        signal_open_ts=signal_open_ts if signal_open_ts is not None else open_ts,
        signal_close_price=signal_close_price if signal_close_price is not None else close_price,
        signal_close_ts=signal_close_ts if signal_close_ts is not None else close_ts,
        open_fill_delay_ticks=open_fill_delay_ticks,
        close_fill_delay_ticks=close_fill_delay_ticks,
    )


def run_backtest(
    df: pd.DataFrame,
    thresh_open_long: float,
    thresh_open_short: float,
    thresh_close_long: float,
    thresh_close_short: float,
    *,
    # --- variation (optimizer search space) ---
    open_frac: float = 1.0,
    close_frac: float = 1.0,
    # --- fixed hyperparams (not in variation at this stage) ---
    max_freshness_ms: Optional[float] = None,
    max_latency_okx_ms: Optional[float] = None,
    max_latency_bybit_ms: Optional[float] = None,
    avg_window_sec: Optional[float] = None,
    # --- gear-1 execution / volume (defaults preserve baseline metric) ---
    Trade_Lat: float = 0.0,
    Check_volume: bool = False,
    position_size: float = 10.0,
    position_frac: float = 1.0,
    fee_rate: float = 0.0,
) -> BacktestResult:
    """
    Historical trade loop for the future optimizer objective.

    Layers
    ------
    Gear-1 baseline: Trade_Lat=0, Check_volume=False, quantity=100, fee_rate=0
      → fill on signal tick, metric = sum(open_spread + close_spread) over closed trades.
    Execution delay: Trade_Lat (ms, equal on both exchanges) shifts fill to the first
      tick with event_local_ts_ms >= signal_ts + Trade_Lat.
    Volume gate: Check_volume (bybit_ws size analogy); False = stub always-pass.

    Parameter contract
    ------------------
    Variation (search space): thresh_open_long/short, thresh_close_long/short,
      open_frac, close_frac.
    Fixed hyperparams (per-run config): max_latency_okx_ms, max_latency_bybit_ms,
      avg_window_sec; optional max_freshness_ms (also fixed, not variation);
      Trade_Lat, Check_volume, position_size, position_frac, fee_rate.

    Open/close if-elif priority is preserved (long open → short open → long close → short close).
    End-of-data open positions are returned for observability but do NOT contribute to metric.
    On each open/close *signal*, stores scalar okx/bybit latency from that row plus a ±5-tick window.

    Optional gates (None = off) only reject candidates; they do not change metric formula
    or branch priority. When a threshold candidate fails a gate, that branch is consumed
    (no fall-through) and counters are updated.

    Gate A — current-tick freshness/latency (per-leg caps):
      max_freshness_ms: both okx_freshness_ms and bybit_freshness_ms <= cap
      max_latency_okx_ms:   okx_latency_ms <= cap_okx   (if set)
      max_latency_bybit_ms: bybit_latency_ms <= cap_bybit (if set)

    Gate B — time-window average (enabled iff avg_window_sec is not None):
      Over [t - avg_window_sec, t] on event_local_ts_ms, average the side's spread
      using only points where (okx <= cap_okx if set) AND (bybit <= cap_bybit if set).
      If neither latency cap is set, all points in the window count.
      Fail-closed if no valid points remain.
      Open:  mean >= open_frac  * thresh_open_*
      Close: mean >= close_frac * thresh_close_*  (symmetric)

    Gate volume — book size on open AND close (enabled iff Check_volume is True):
      Requires parquet cols okx_bid_size / okx_ask_size / bybit_bid_size / bybit_ask_size.
      Missing columns → ValueError (not silent pass).
      Analogy to else/bybit_ws.py: Bybit size > position_size,
      OKX size > position_size/10 on the traded sides. Check_volume=False → always pass.

    Execution delay (causal only, equal on both exchanges):
      Trade_Lat > 0 → fill price/ts from first tick with ts >= signal_ts + Trade_Lat.
      Signal still gates at signal tick; pending fill blocks new signals until resolved.
      If fill index falls past series end, the pending action is dropped (n_pending_missed).

    Metric delta vs baseline when realism off: none (pnl = open+close, fees=0, qty=100).
    With position_frac<1: pnl scales by frac. With fee_rate>0: subtract 4*fee_rate*100*frac
    round-trip (2 legs open + 2 close). With delay: fill spreads may differ from signal.
    """
    if avg_window_sec is not None and avg_window_sec <= 0:
        raise ValueError("avg_window_sec must be > 0 when set")
    if not (0.0 < open_frac <= 1.0):
        raise ValueError("open_frac must be in (0, 1]")
    if not (0.0 < close_frac <= 1.0):
        raise ValueError("close_frac must be in (0, 1]")
    if not (0.0 < position_frac <= 1.0):
        raise ValueError("position_frac must be in (0, 1]")
    if fee_rate < 0.0:
        raise ValueError("fee_rate must be >= 0")
    if float(Trade_Lat) < 0.0:
        raise ValueError("Trade_Lat must be >= 0")
    if float(position_size) <= 0.0:
        raise ValueError("position_size must be > 0")
    if Check_volume:
        _require_size_columns(df)

    target_qty = 100.0 * float(position_frac)
    data = df.sort_values("event_dt").reset_index(drop=True)

    # Column presence when gates are enabled (fail loud, not silent).
    if max_freshness_ms is not None:
        for col in ("okx_freshness_ms", "bybit_freshness_ms"):
            if col not in data.columns:
                raise KeyError(f"Gate A freshness requires column {col!r}")
    need_latency_cols = (
        max_latency_okx_ms is not None
        or max_latency_bybit_ms is not None
        or avg_window_sec is not None
    )
    if need_latency_cols:
        for col in ("okx_latency_ms", "bybit_latency_ms"):
            if col not in data.columns:
                raise KeyError(f"latency/avg gates require column {col!r}")
    # Check_volume size columns already validated by _require_size_columns above.

    ts_ms = data["event_local_ts_ms"].to_numpy(dtype="float64", copy=False)
    spread_long_arr = data["spread_long"].to_numpy(dtype="float64", copy=False)
    spread_short_arr = data["spread_short"].to_numpy(dtype="float64", copy=False)
    okx_lat_arr = data["okx_latency_ms"].to_numpy(dtype="float64", copy=False)
    bybit_lat_arr = data["bybit_latency_ms"].to_numpy(dtype="float64", copy=False)
    if max_freshness_ms is not None:
        okx_fresh_arr = data["okx_freshness_ms"].to_numpy(dtype="float64", copy=False)
        bybit_fresh_arr = data["bybit_freshness_ms"].to_numpy(dtype="float64", copy=False)
    else:
        okx_fresh_arr = bybit_fresh_arr = None

    if Check_volume:
        size_arrs = {
            col: data[col].to_numpy(dtype="float64", copy=False) for col in BOOK_SIZE_COLS
        }
    else:
        size_arrs = None

    # Valid-for-average mask: each set cap filters its leg; unset cap → no filter on that leg.
    # NaN comparisons are False → excluded (fail-closed for that point).
    if avg_window_sec is not None:
        avg_valid = np.ones(len(data), dtype=bool)
        if max_latency_okx_ms is not None:
            avg_valid &= okx_lat_arr <= max_latency_okx_ms
        if max_latency_bybit_ms is not None:
            avg_valid &= bybit_lat_arr <= max_latency_bybit_ms
    else:
        avg_valid = None

    # Sliding window state over sorted event_local_ts_ms (O(n) two pointers).
    # Maintains running sums of latency-valid points in [t - W, t].
    window_ms = (
        float(avg_window_sec) * 1000.0 if avg_window_sec is not None else None
    )
    left = 0
    sum_long = 0.0
    sum_short = 0.0
    win_count = 0

    def _advance_window(i: int) -> None:
        nonlocal left, sum_long, sum_short, win_count
        assert window_ms is not None and avg_valid is not None
        t_i = ts_ms[i]
        t_lo = t_i - window_ms
        while left <= i and ts_ms[left] < t_lo:
            if avg_valid[left]:
                sum_long -= spread_long_arr[left]
                sum_short -= spread_short_arr[left]
                win_count -= 1
            left += 1
        # Include i once (windows are advanced strictly forward).
        if avg_valid[i]:
            sum_long += spread_long_arr[i]
            sum_short += spread_short_arr[i]
            win_count += 1

    def _mean_in_window(side_spread: str) -> Optional[float]:
        if win_count <= 0:
            return None
        if side_spread == "long":
            return sum_long / win_count
        return sum_short / win_count

    def _gate_a_ok(i: int) -> tuple[bool, Optional[str]]:
        """Return (ok, fail_reason) for Gate A on tick i."""
        if max_freshness_ms is not None:
            assert okx_fresh_arr is not None and bybit_fresh_arr is not None
            of_ = okx_fresh_arr[i]
            bf_ = bybit_fresh_arr[i]
            if of_ != of_ or bf_ != bf_ or of_ > max_freshness_ms or bf_ > max_freshness_ms:
                return False, "freshness"
        # Per-leg latency: each exchange checked against its own cap.
        if max_latency_okx_ms is not None:
            ol_ = okx_lat_arr[i]
            if ol_ != ol_ or ol_ > max_latency_okx_ms:
                return False, "latency"
        if max_latency_bybit_ms is not None:
            bl_ = bybit_lat_arr[i]
            if bl_ != bl_ or bl_ > max_latency_bybit_ms:
                return False, "latency"
        return True, None

    def _gate_b_ok(mean_val: Optional[float], frac: float, thresh: float) -> bool:
        if avg_window_sec is None:
            return True
        if mean_val is None:
            return False  # fail-closed: empty latency-filtered window
        return mean_val >= frac * thresh

    def _gate_volume_ok(i: int, side: Side, *, is_open: bool) -> bool:
        if not Check_volume:
            return True
        assert size_arrs is not None
        return _volume_ok_bybit_ws(
            size_arrs, i, side, is_open=is_open, position_size=position_size
        )

    trades: list[Trade] = []
    metric = 0.0
    fees_total = 0.0
    pos = 0.0
    whatpos: Optional[Side] = None
    open_price: Optional[float] = None
    open_ts: Optional[int] = None
    open_dt = None
    open_okx_lat: Optional[float] = None
    open_bybit_lat: Optional[float] = None
    open_okx_win: Optional[list[Optional[float]]] = None
    open_bybit_win: Optional[list[Optional[float]]] = None
    open_win_offsets: Optional[list[int]] = None
    open_win_trunc: bool = False
    signal_open_price: Optional[float] = None
    signal_open_ts: Optional[int] = None
    open_fill_delay_ticks: int = 0
    open_fees: float = 0.0

    # Pending causal fill: {"kind","side","signal_i","fill_i", ...signal snapshots}
    pending: Optional[dict] = None

    n_signals_raw = 0
    n_signals_passed = 0
    n_filtered_by_freshness = 0
    n_filtered_by_latency = 0
    n_filtered_by_avg = 0
    n_filtered_by_size = 0
    n_pending_missed = 0

    def _schedule_or_fill(kind: str, side: Side, signal_i: int, row) -> None:
        """Pass gates → either fill immediately or schedule causal delayed fill."""
        nonlocal pending, n_pending_missed, n_signals_passed
        fill_i = _resolve_fill_index(
            signal_i,
            ts_ms,
            Trade_Lat=Trade_Lat,
        )
        if fill_i is None:
            n_pending_missed += 1
            return
        n_signals_passed += 1
        payload = {
            "kind": kind,
            "side": side,
            "signal_i": signal_i,
            "fill_i": fill_i,
            "signal_row_spread_long": float(row.spread_long),
            "signal_row_spread_short": float(row.spread_short),
            "signal_ts": int(row.event_local_ts_ms),
            "signal_dt": row.event_dt,
            "signal_okx_lat": float(row.okx_latency_ms),
            "signal_bybit_lat": float(row.bybit_latency_ms),
        }
        if fill_i == signal_i:
            _execute_fill(payload, signal_i, row)
        else:
            pending = payload

    def _execute_fill(payload: dict, fill_i: int, fill_row) -> None:
        nonlocal pos, whatpos, open_price, open_ts, open_dt
        nonlocal open_okx_lat, open_bybit_lat, open_okx_win, open_bybit_win
        nonlocal open_win_offsets, open_win_trunc
        nonlocal signal_open_price, signal_open_ts, open_fill_delay_ticks, open_fees
        nonlocal metric, fees_total

        kind = payload["kind"]
        side: Side = payload["side"]
        signal_i = int(payload["signal_i"])
        delay_ticks = fill_i - signal_i

        if kind == "open":
            if side == "long":
                fill_spread = float(fill_row.spread_long)
                sig_spread = float(payload["signal_row_spread_long"])
            else:
                fill_spread = float(fill_row.spread_short)
                sig_spread = float(payload["signal_row_spread_short"])
            pos = target_qty
            open_price = fill_spread
            open_ts = int(fill_row.event_local_ts_ms)
            open_dt = fill_row.event_dt
            open_okx_lat = float(payload["signal_okx_lat"])
            open_bybit_lat = float(payload["signal_bybit_lat"])
            open_okx_win, open_bybit_win, open_win_offsets, open_win_trunc = (
                _latency_window_at(data, signal_i)
            )
            signal_open_price = sig_spread
            signal_open_ts = int(payload["signal_ts"])
            open_fill_delay_ticks = delay_ticks
            open_fees = _fee_cost_pct(target_qty, fee_rate, legs=2)
            whatpos = side
            return

        # close
        assert open_price is not None and open_ts is not None and whatpos == side
        if side == "long":
            fill_spread = float(fill_row.spread_short)
            sig_spread = float(payload["signal_row_spread_short"])
        else:
            fill_spread = float(fill_row.spread_long)
            sig_spread = float(payload["signal_row_spread_long"])
        close_okx_win, close_bybit_win, close_win_offsets, close_win_trunc = (
            _latency_window_at(data, signal_i)
        )
        close_fees = _fee_cost_pct(pos, fee_rate, legs=2)
        fees = open_fees + close_fees
        trade = _closed_trade(
            side=side,
            open_price=open_price,
            open_ts=open_ts,
            open_dt=open_dt,
            close_price=fill_spread,
            close_ts=int(fill_row.event_local_ts_ms),
            close_dt=fill_row.event_dt,
            quantity=pos,
            fees=fees,
            okx_latency_ms_open=open_okx_lat,
            bybit_latency_ms_open=open_bybit_lat,
            okx_latency_ms_close=float(payload["signal_okx_lat"]),
            bybit_latency_ms_close=float(payload["signal_bybit_lat"]),
            okx_latency_ms_open_window=open_okx_win,
            bybit_latency_ms_open_window=open_bybit_win,
            latency_window_offsets_open=open_win_offsets,
            latency_window_truncated_open=open_win_trunc,
            okx_latency_ms_close_window=close_okx_win,
            bybit_latency_ms_close_window=close_bybit_win,
            latency_window_offsets_close=close_win_offsets,
            latency_window_truncated_close=close_win_trunc,
            signal_open_price=signal_open_price,
            signal_open_ts=signal_open_ts,
            signal_close_price=sig_spread,
            signal_close_ts=int(payload["signal_ts"]),
            open_fill_delay_ticks=open_fill_delay_ticks,
            close_fill_delay_ticks=delay_ticks,
        )
        trades.append(trade)
        metric += float(trade.pnl)
        fees_total += fees
        pos = 0.0
        whatpos = None
        open_price = None
        open_ts = None
        open_dt = None
        open_okx_lat = None
        open_bybit_lat = None
        open_okx_win = None
        open_bybit_win = None
        open_win_offsets = None
        open_win_trunc = False
        signal_open_price = None
        signal_open_ts = None
        open_fill_delay_ticks = 0
        open_fees = 0.0

    n = len(data)
    for i, row in enumerate(data.itertuples(index=False)):
        if window_ms is not None:
            _advance_window(i)

        # Resolve pending causal fill before considering new signals.
        if pending is not None and i == int(pending["fill_i"]):
            _execute_fill(pending, i, row)
            pending = None

        # Block new signals while a delayed fill is outstanding.
        if pending is not None:
            continue

        # --- candidates (threshold only); gates reject inside the chosen branch ---
        if row.spread_long > thresh_open_long and pos < target_qty:
            n_signals_raw += 1
            ok_a, reason = _gate_a_ok(i)
            if not ok_a:
                if reason == "freshness":
                    n_filtered_by_freshness += 1
                else:
                    n_filtered_by_latency += 1
            else:
                mean_l = (
                    _mean_in_window("long") if avg_window_sec is not None else None
                )
                if not _gate_b_ok(mean_l, open_frac, thresh_open_long):
                    n_filtered_by_avg += 1
                elif not _gate_volume_ok(i, "long", is_open=True):
                    n_filtered_by_size += 1
                else:
                    _schedule_or_fill("open", "long", i, row)
        elif row.spread_short > thresh_open_short and pos < target_qty:
            n_signals_raw += 1
            ok_a, reason = _gate_a_ok(i)
            if not ok_a:
                if reason == "freshness":
                    n_filtered_by_freshness += 1
                else:
                    n_filtered_by_latency += 1
            else:
                mean_s = (
                    _mean_in_window("short") if avg_window_sec is not None else None
                )
                if not _gate_b_ok(mean_s, open_frac, thresh_open_short):
                    n_filtered_by_avg += 1
                elif not _gate_volume_ok(i, "short", is_open=True):
                    n_filtered_by_size += 1
                else:
                    _schedule_or_fill("open", "short", i, row)
        elif row.spread_short > thresh_close_long and pos > 0 and whatpos == "long":
            n_signals_raw += 1
            ok_a, reason = _gate_a_ok(i)
            if not ok_a:
                if reason == "freshness":
                    n_filtered_by_freshness += 1
                else:
                    n_filtered_by_latency += 1
            else:
                # Close long uses spread_short (same leg as the close threshold).
                mean_s = (
                    _mean_in_window("short") if avg_window_sec is not None else None
                )
                if not _gate_b_ok(mean_s, close_frac, thresh_close_long):
                    n_filtered_by_avg += 1
                elif not _gate_volume_ok(i, "long", is_open=False):
                    n_filtered_by_size += 1
                else:
                    _schedule_or_fill("close", "long", i, row)
        elif row.spread_long > thresh_close_short and pos > 0 and whatpos == "short":
            n_signals_raw += 1
            ok_a, reason = _gate_a_ok(i)
            if not ok_a:
                if reason == "freshness":
                    n_filtered_by_freshness += 1
                else:
                    n_filtered_by_latency += 1
            else:
                mean_l = (
                    _mean_in_window("long") if avg_window_sec is not None else None
                )
                if not _gate_b_ok(mean_l, close_frac, thresh_close_short):
                    n_filtered_by_avg += 1
                elif not _gate_volume_ok(i, "short", is_open=False):
                    n_filtered_by_size += 1
                else:
                    _schedule_or_fill("close", "short", i, row)

    if pending is not None:
        n_pending_missed += 1
        pending = None

    open_position: Optional[Trade] = None
    if whatpos is not None and open_price is not None and open_ts is not None:
        open_position = Trade(
            side=whatpos,
            status="open",
            open_price=open_price,
            open_ts=open_ts,
            open_dt=open_dt,
            quantity=pos,
            fees=open_fees,
            okx_latency_ms_open=open_okx_lat,
            bybit_latency_ms_open=open_bybit_lat,
            okx_latency_ms_open_window=open_okx_win,
            bybit_latency_ms_open_window=open_bybit_win,
            latency_window_offsets_open=open_win_offsets,
            latency_window_truncated_open=open_win_trunc,
            signal_open_price=signal_open_price,
            signal_open_ts=signal_open_ts,
            open_fill_delay_ticks=open_fill_delay_ticks,
        )
        trades.append(open_position)

    return BacktestResult(
        trades=trades,
        metric=metric,
        open_position=open_position,
        n_signals_raw=n_signals_raw,
        n_signals_passed=n_signals_passed,
        n_filtered_by_freshness=n_filtered_by_freshness,
        n_filtered_by_latency=n_filtered_by_latency,
        n_filtered_by_avg=n_filtered_by_avg,
        n_filtered_by_size=n_filtered_by_size,
        n_pending_missed=n_pending_missed,
        fees_total=fees_total,
    )


# Backward-compatible alias (same thresholds / same objective)
def trademodel(
    thresh_open_long: float,
    thresh_open_short: float,
    thresh_close_long: float,
    thresh_close_short: float,
) -> tuple[list[Trade], float]:
    result = run_backtest(
        df,
        thresh_open_long,
        thresh_open_short,
        thresh_close_long,
        thresh_close_short,
        open_frac=open_frac,
        close_frac=close_frac,
        max_freshness_ms=max_freshness_ms,
        max_latency_okx_ms=max_latency_okx_ms,
        max_latency_bybit_ms=max_latency_bybit_ms,
        avg_window_sec=avg_window_sec,
        Trade_Lat=Trade_Lat,
        Check_volume=Check_volume,
        position_size=position_size,
        position_frac=position_frac,
        fee_rate=fee_rate,
    )
    return result.trades, result.metric


def run_backtest_from_params(
    df: pd.DataFrame,
    variation: dict,
    hyperparams: dict,
) -> BacktestResult:
    """Thin wrapper: variation vs fixed hyperparams for a future objective(params).

    variation keys: thresh_open_long/short, thresh_close_long/short, open_frac, close_frac
    hyperparams keys: max_latency_okx_ms, max_latency_bybit_ms, avg_window_sec,
                      optional max_freshness_ms,
                      Trade_Lat, Check_volume, position_size, position_frac, fee_rate
    """
    return run_backtest(
        df,
        variation["thresh_open_long"],
        variation["thresh_open_short"],
        variation["thresh_close_long"],
        variation["thresh_close_short"],
        open_frac=variation.get("open_frac", 1.0),
        close_frac=variation.get("close_frac", 1.0),
        max_freshness_ms=hyperparams.get("max_freshness_ms"),
        max_latency_okx_ms=hyperparams.get("max_latency_okx_ms"),
        max_latency_bybit_ms=hyperparams.get("max_latency_bybit_ms"),
        avg_window_sec=hyperparams.get("avg_window_sec"),
        Trade_Lat=hyperparams.get("Trade_Lat", 0.0),
        Check_volume=hyperparams.get("Check_volume", False),
        position_size=hyperparams.get("position_size", 10.0),
        position_frac=hyperparams.get("position_frac", variation.get("position_frac", 1.0)),
        fee_rate=hyperparams.get("fee_rate", 0.0),
    )


In [ ]:
def _downsample_for_plot(
    data: pd.DataFrame,
    max_points: Optional[int] = None,
) -> pd.DataFrame:
    """
    Thin spread series for plotting only.

    max_points: if set and len(data) > max_points, keep every N-th row
    (uniform stride). Trade markers are NOT downsampled — callers pass
    trades separately on full timestamps. Pass None to keep all rows.
    """
    if max_points is None or max_points <= 0 or len(data) <= max_points:
        return data
    step = max(1, len(data) // max_points)
    # Always keep first/last row so the day edges stay visible.
    idx = list(range(0, len(data), step))
    if idx[-1] != len(data) - 1:
        idx.append(len(data) - 1)
    return data.iloc[idx]


def plot_coin_spreads(
    df: pd.DataFrame,
    *,
    max_points: Optional[int] = None,
    title: Optional[str] = None,
) -> go.Figure:
    """Historical spread series only (no trade markers)."""
    return plot_strategy(df, trades=None, max_points=max_points, title=title)


def plot_strategy(
    df: pd.DataFrame,
    trades: Optional[list[Trade]] = None,
    *,
    title: Optional[str] = None,
    width: int = 1400,
    height: int = 700,
    max_points: Optional[int] = None,
) -> go.Figure:
    """
    Plot spread_long / spread_short with entry/exit markers.

    max_points: downsample line traces to at most this many points
    (default: notebook config `max_points`). Trade markers always use
    full open/close timestamps — only the continuous spread lines are thinned.
    Uses Scattergl for lines to keep day-long series responsive.

    Returns the Figure; do not call fig.show() here — Jupyter displays the
    return value once when the call is the last expression in a cell.
    """
    if max_points is None:
        max_points = globals().get("max_points", 4000)
    if title is None:
        coin = globals().get("base_coin", "?")
        day = globals().get("event_date", "?")
        title = f"{coin} {day} — spreads + entries/exits"

    data = df.sort_values("event_dt")
    plot_data = _downsample_for_plot(data, max_points=max_points)
    fig = go.Figure()

    # WebGL lines: cheaper for large day-long series after downsample.
    fig.add_trace(
        go.Scattergl(
            x=plot_data["event_dt"],
            y=plot_data["spread_long"],
            mode="lines",
            name="spread_long",
            line=dict(width=1.4, color="#1f77b4"),
        )
    )
    fig.add_trace(
        go.Scattergl(
            x=plot_data["event_dt"],
            y=plot_data["spread_short"],
            mode="lines",
            name="spread_short",
            line=dict(width=1.4, color="#d62728"),
        )
    )

    if trades:
        marker_specs = {
            ("long", "open"): dict(name="long open", symbol="triangle-up", color="#2ca02c", size=11),
            ("long", "close"): dict(name="long close", symbol="triangle-down", color="#1f77b4", size=11),
            ("short", "open"): dict(name="short open", symbol="triangle-up", color="#ff7f0e", size=11),
            ("short", "close"): dict(name="short close", symbol="triangle-down", color="#d62728", size=11),
            ("long", "open_eod"): dict(name="long open (EOD)", symbol="diamond", color="#2ca02c", size=13),
            ("short", "open_eod"): dict(name="short open (EOD)", symbol="diamond", color="#ff7f0e", size=13),
        }
        buckets: dict[tuple[str, str], list[Trade]] = {key: [] for key in marker_specs}

        for trade in trades:
            if trade.status == "open":
                buckets[(trade.side, "open_eod")].append(trade)
            else:
                buckets[(trade.side, "open")].append(trade)
                buckets[(trade.side, "close")].append(trade)

        for key, group in buckets.items():
            if not group:
                continue
            spec = marker_specs[key]
            event = key[1]
            if event in ("open", "open_eod"):
                xs = [t.open_dt for t in group]
                ys = [t.open_price for t in group]
                hover = [
                    (
                        f"{t.side} {t.status}"
                        f"<br>open={t.open_price:.6f}<br>ts={t.open_ts}"
                        f"<br>okx_lat_open={t.okx_latency_ms_open}"
                        f"<br>bybit_lat_open={t.bybit_latency_ms_open}"
                    )
                    for t in group
                ]
            else:
                xs = [t.close_dt for t in group]
                ys = [t.close_price for t in group]
                hover = [
                    (
                        f"{t.side} close<br>close={t.close_price:.6f}<br>pnl={t.pnl:.6f}"
                        f"<br>okx_lat_close={t.okx_latency_ms_close}"
                        f"<br>bybit_lat_close={t.bybit_latency_ms_close}"
                    )
                    for t in group
                ]
            # Markers stay on SVG Scatter (few points, readable symbols).
            fig.add_trace(
                go.Scatter(
                    x=xs,
                    y=ys,
                    mode="markers",
                    name=spec["name"],
                    marker=dict(
                        symbol=spec["symbol"],
                        color=spec["color"],
                        size=spec["size"],
                        line=dict(width=1, color="#222"),
                    ),
                    text=hover,
                    hovertemplate="%{text}<extra></extra>",
                )
            )

    fig.update_layout(
        title=title,
        width=width,
        height=height,
        xaxis_title="event_dt",
        yaxis_title="spread",
        hovermode="x unified",
        legend=dict(font=dict(size=11)),
        margin=dict(l=50, r=20, t=60, b=40),
    )
    # No fig.show() — avoids double-render with Jupyter's display of the return value.
    return fig


# Demo: ungated vs gated (same thresholds from VARIATION defaults)
THRESH = (
    thresh_open_long,
    thresh_open_short,
    thresh_close_long,
    thresh_close_short,
)

result_off = run_backtest(df, *THRESH)
result_on = run_backtest(
    df,
    *THRESH,
    open_frac=demo_open_frac,
    close_frac=demo_close_frac,
    max_freshness_ms=demo_max_freshness_ms,
    max_latency_okx_ms=demo_max_latency_okx_ms,
    max_latency_bybit_ms=demo_max_latency_bybit_ms,
    avg_window_sec=demo_avg_window_sec,
)

# Gear-1 with Trade_Lat + optional volume (defaults Check_volume=False preserve path).
result_realism = run_backtest(
    df,
    *THRESH,
    open_frac=demo_open_frac,
    close_frac=demo_close_frac,
    max_freshness_ms=demo_max_freshness_ms,
    max_latency_okx_ms=demo_max_latency_okx_ms,
    max_latency_bybit_ms=demo_max_latency_bybit_ms,
    avg_window_sec=demo_avg_window_sec,
    Trade_Lat=demo_Trade_Lat,
    Check_volume=demo_Check_volume,
    position_size=position_size,
    position_frac=demo_position_frac,
    fee_rate=demo_fee_rate,
)


def _summarize(label: str, r: BacktestResult) -> None:
    closed_n = sum(1 for t in r.trades if t.status == "closed")
    open_n = 1 if r.open_position is not None else 0
    print(
        f"{label}: metric={r.metric:.6f}  closed={closed_n}  open_at_end={open_n}  "
        f"raw={r.n_signals_raw}  passed={r.n_signals_passed}  "
        f"filt_fresh={r.n_filtered_by_freshness}  filt_lat={r.n_filtered_by_latency}  "
        f"filt_avg={r.n_filtered_by_avg}  filt_size={r.n_filtered_by_size}  "
        f"pending_miss={r.n_pending_missed}  fees={r.fees_total:.6f}"
    )


print(
    f"gates ON params: freshness<={demo_max_freshness_ms}ms  "
    f"lat_okx<={demo_max_latency_okx_ms}ms  lat_bybit<={demo_max_latency_bybit_ms}ms  "
    f"avg_window={demo_avg_window_sec}s  open_frac={demo_open_frac}  "
    f"close_frac={demo_close_frac}"
)
_summarize("OFF (no gates)", result_off)
_summarize("ON  (gates)   ", result_on)
print(
    f"Trade_Lat/volume ON: Trade_Lat={demo_Trade_Lat}  Check_volume={demo_Check_volume}  "
    f"position_frac={demo_position_frac}  fee_rate={demo_fee_rate}"
)
_summarize("ON+Trade_Lat ", result_realism)

# Sanity: Trade_Lat=0 + Check_volume=False must match gated-only metric (baseline path).
result_realism_off = run_backtest(
    df,
    *THRESH,
    open_frac=demo_open_frac,
    close_frac=demo_close_frac,
    max_freshness_ms=demo_max_freshness_ms,
    max_latency_okx_ms=demo_max_latency_okx_ms,
    max_latency_bybit_ms=demo_max_latency_bybit_ms,
    avg_window_sec=demo_avg_window_sec,
    Trade_Lat=0.0,
    Check_volume=False,
    position_frac=1.0,
    fee_rate=0.0,
)
assert abs(result_realism_off.metric - result_on.metric) < 1e-9, (
    result_realism_off.metric,
    result_on.metric,
)
print("ok: Trade_Lat=0 + Check_volume=False == gated baseline metric")

# Main config path: Trade_Lat from cell 1 (equal delay both exchanges), volume stub off.
result_gear1 = run_backtest(
    df,
    *THRESH,
    open_frac=open_frac,
    close_frac=close_frac,
    max_freshness_ms=max_freshness_ms,
    max_latency_okx_ms=max_latency_okx_ms,
    max_latency_bybit_ms=max_latency_bybit_ms,
    avg_window_sec=avg_window_sec,
    Trade_Lat=Trade_Lat,
    Check_volume=Check_volume,
    position_size=position_size,
    position_frac=position_frac,
    fee_rate=fee_rate,
)
_summarize("gear1 config ", result_gear1)

# Plot uses gear1 config (Trade_Lat); swap to result_off / result_on / result_realism to compare.
result = result_gear1
plot_strategy(
    df,
    result.trades,
    title=(
        f"{base_coin} {event_date} — gear1 Trade_Lat={Trade_Lat}ms "
        f"Check_volume={Check_volume}"
    ),
    max_points=max_points,
)


In [ ]:
def plot_latencies(
    df: pd.DataFrame,
    trades: Optional[list[Trade]] = None,
    *,
    title: Optional[str] = None,
    width: int = 1400,
    height: int = 500,
    max_points: Optional[int] = None,
    show_trade_markers: bool = True,
) -> go.Figure:
    """
    Time series of okx_latency_ms / bybit_latency_ms (parquet delivery latency).

    max_points: downsample line traces the same way as spread plots.
    Trade markers (optional) use full open/close timestamps and y = that
    exchange's latency snapshot stored on the Trade at signal time.
    """
    if max_points is None:
        max_points = globals().get("max_points", 4000)
    if title is None:
        coin = globals().get("base_coin", "?")
        day = globals().get("event_date", "?")
        title = f"{coin} {day} — delivery latency (ms)"

    data = df.sort_values("event_dt")
    plot_data = _downsample_for_plot(data, max_points=max_points)
    fig = go.Figure()

    fig.add_trace(
        go.Scattergl(
            x=plot_data["event_dt"],
            y=plot_data["okx_latency_ms"],
            mode="lines",
            name="okx_latency_ms",
            line=dict(width=1.2, color="#9467bd"),
        )
    )
    fig.add_trace(
        go.Scattergl(
            x=plot_data["event_dt"],
            y=plot_data["bybit_latency_ms"],
            mode="lines",
            name="bybit_latency_ms",
            line=dict(width=1.2, color="#8c564b"),
        )
    )

    if show_trade_markers and trades:
        # Markers at signal time; y = latency of the triggering tick (per exchange).
        open_okx_x, open_okx_y, open_okx_h = [], [], []
        open_by_x, open_by_y, open_by_h = [], [], []
        close_okx_x, close_okx_y, close_okx_h = [], [], []
        close_by_x, close_by_y, close_by_h = [], [], []

        for t in trades:
            open_okx_x.append(t.open_dt)
            open_okx_y.append(t.okx_latency_ms_open)
            open_okx_h.append(f"{t.side} open | okx_lat={t.okx_latency_ms_open}")
            open_by_x.append(t.open_dt)
            open_by_y.append(t.bybit_latency_ms_open)
            open_by_h.append(f"{t.side} open | bybit_lat={t.bybit_latency_ms_open}")
            if t.status == "closed":
                close_okx_x.append(t.close_dt)
                close_okx_y.append(t.okx_latency_ms_close)
                close_okx_h.append(f"{t.side} close | okx_lat={t.okx_latency_ms_close}")
                close_by_x.append(t.close_dt)
                close_by_y.append(t.bybit_latency_ms_close)
                close_by_h.append(f"{t.side} close | bybit_lat={t.bybit_latency_ms_close}")

        marker_traces = [
            ("open @ okx lat", open_okx_x, open_okx_y, open_okx_h, "triangle-up", "#9467bd"),
            ("open @ bybit lat", open_by_x, open_by_y, open_by_h, "triangle-up", "#8c564b"),
            ("close @ okx lat", close_okx_x, close_okx_y, close_okx_h, "triangle-down", "#9467bd"),
            ("close @ bybit lat", close_by_x, close_by_y, close_by_h, "triangle-down", "#8c564b"),
        ]
        for name, xs, ys, hover, symbol, color in marker_traces:
            if not xs:
                continue
            fig.add_trace(
                go.Scatter(
                    x=xs,
                    y=ys,
                    mode="markers",
                    name=name,
                    marker=dict(symbol=symbol, color=color, size=10, line=dict(width=1, color="#222")),
                    text=hover,
                    hovertemplate="%{text}<extra></extra>",
                )
            )

    fig.update_layout(
        title=title,
        width=width,
        height=height,
        xaxis_title="event_dt",
        yaxis_title="latency_ms",
        hovermode="x unified",
        legend=dict(font=dict(size=11)),
        margin=dict(l=50, r=20, t=60, b=40),
    )
    return fig


def plot_latency_hist(
    df: pd.DataFrame,
    *,
    title: Optional[str] = None,
    width: int = 1000,
    height: int = 450,
    nbins: Optional[int] = None,
) -> go.Figure:
    """
    Overlay histogram of okx_latency_ms and bybit_latency_ms for the day.

    nbins: bin count (default: notebook config `latency_hist_bins`).
    """
    if nbins is None:
        nbins = int(globals().get("latency_hist_bins", 60))
    if title is None:
        coin = globals().get("base_coin", "?")
        day = globals().get("event_date", "?")
        title = f"{coin} {day} — latency histogram (ms)"

    fig = go.Figure()
    fig.add_trace(
        go.Histogram(
            x=df["okx_latency_ms"],
            name="okx_latency_ms",
            nbinsx=nbins,
            opacity=0.55,
            marker_color="#9467bd",
        )
    )
    fig.add_trace(
        go.Histogram(
            x=df["bybit_latency_ms"],
            name="bybit_latency_ms",
            nbinsx=nbins,
            opacity=0.55,
            marker_color="#8c564b",
        )
    )
    fig.update_layout(
        title=title,
        width=width,
        height=height,
        barmode="overlay",
        xaxis_title="latency_ms",
        yaxis_title="count",
        legend=dict(font=dict(size=11)),
        margin=dict(l=50, r=20, t=60, b=40),
    )
    return fig


# Demo: latency viz (reuses result from previous cell if present)
if "result" not in globals():
    result = run_backtest(df, 0.5, 0.5, 0.5, 0.5)

# Sample trade latencies + ±5 tick windows
for t in result.trades[:3]:
    print(
        f"{t.side}/{t.status}: "
        f"open okx={t.okx_latency_ms_open} bybit={t.bybit_latency_ms_open} | "
        f"close okx={t.okx_latency_ms_close} bybit={t.bybit_latency_ms_close}"
    )

t0 = result.trades[0]
print(
    f"\nwindow demo ({t0.side}/{t0.status}): "
    f"offsets_open={t0.latency_window_offsets_open} "
    f"truncated_open={t0.latency_window_truncated_open}"
)
print(f"  okx_open_window={t0.okx_latency_ms_open_window}")
print(f"  bybit_open_window={t0.bybit_latency_ms_open_window}")
if t0.status == "closed":
    print(
        f"  offsets_close={t0.latency_window_offsets_close} "
        f"truncated_close={t0.latency_window_truncated_close}"
    )
    print(f"  okx_close_window={t0.okx_latency_ms_close_window}")
    print(f"  bybit_close_window={t0.bybit_latency_ms_close_window}")

plot_latencies(
    df,
    result.trades,
    title=f"{base_coin} {event_date} — delivery latency",
    max_points=max_points,
)


In [ ]:
plot_latency_hist(
    df,
    title=f"{base_coin} {event_date} — latency histogram",
    nbins=latency_hist_bins,
)
